# Model Comparison with considerations for Edge Deployment

This notebook compares lightweight CNN architectures to determine the best model for 
driver‑behavior classification under edge‑computing constraints.

Architectures evaluated:
- MobileNetV3‑Small
- EfficientNet‑B0
- ShuffleNetV2
- ResNet18 (baseline)


In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
import time
import pandas as pd

from pathlib import Path
import sys

sys.path.append("..")
from dataset import DriverDataset, get_transforms


## Load Dataset

In [14]:
from torch.utils.data import random_split
train_tf, val_tf = get_transforms()

#train_ds = DriverDataset("../data/imgs/train", transform=train_tf)
#val_ds   = DriverDataset("../data/imgs/train", transform=val_tf)

full_ds = DriverDataset("../data/imgs/train", transform=None)

train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size

train_ds, val_ds = random_split(full_ds, [train_size, val_size])
train_ds.dataset.transform = train_tf
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)


## Define Models

In [3]:
def build_mobilenet():
    m = models.mobilenet_v3_small(weights="IMAGENET1K_V1")
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 10)
    return m

def build_efficientnet():
    m = models.efficientnet_b0(weights="IMAGENET1K_V1")
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 10)
    return m

def build_shufflenet():
    m = models.shufflenet_v2_x1_0(weights="IMAGENET1K_V1")
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m

def build_resnet18():
    m = models.resnet18(weights="IMAGENET1K_V1")
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m


## Compare Model Complexity

In [4]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

models_to_test = {
    "MobileNetV3-Small": build_mobilenet(),
    "EfficientNet-B0": build_efficientnet(),
    "ShuffleNetV2": build_shufflenet(),
    "ResNet18": build_resnet18()
}

for name, model in models_to_test.items():
    print(name, "params:", count_params(model))


MobileNetV3-Small params: 1528106
EfficientNet-B0 params: 4020358
ShuffleNetV2 params: 1263854
ResNet18 params: 11181642


## Inference Latency Benchmark

In [16]:
def benchmark(model, device="cpu", runs=50):
    model.eval().to(device)
    dummy = torch.randn(1, 3, 224, 224).to(device)

    # warmup
    for _ in range(10):
        _ = model(dummy)

    start = time.time()
    for _ in range(runs):
        _ = model(dummy)
    end = time.time()

    return (end - start) / runs

latencies = {}
for name, model in models_to_test.items():
    latencies[name] = benchmark(model)

latencies


{'MobileNetV3-Small': 0.005876679420471192,
 'EfficientNet-B0': 0.015416488647460938,
 'ShuffleNetV2': 0.010135188102722167,
 'ResNet18': 0.010062394142150878}

Latency somewhat aligns with model complexity, EfficientNet-B0 being more complex and slightly slower than others. Standout is MobileNetV3-Small, with much lower latency despite similar complexity to ResNet18 and ShuffleNetV2.

## Quick Training Runs

In [10]:
def quick_train(model, epochs=2):
    model = model.to("cuda" if torch.cuda.is_available() else "cpu")
    device = next(model.parameters()).device

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            out = model(imgs)
            loss = loss_fn(out, labels)
            loss.backward()
            opt.step()

    # quick validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [15]:
results = []

for name, model in models_to_test.items():
    acc = quick_train(model, epochs=2)
    results.append({
        "model": name,
        "params": count_params(model),
        "latency_ms": latencies[name] * 1000,
        "val_acc_2_epochs": acc
    })

pd.DataFrame(results)


,model,params,latency_ms,val_acc_2_epochs
0,MobileNetV3-Small,1528106,5.793118,0.987291
1,EfficientNet-B0,4020358,15.456071,0.995541
2,ShuffleNetV2,1263854,10.001755,0.993980
3,ResNet18,11181642,9.959331,0.991081


Strong results from each model, but significantly lower latency for MobileNetV3-Small, hence best suited for edge applications.